In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import OneClassSVM
import joblib
from src.ovo_svm import OvO_SVM
from utils.Hog import calc_gradients, HoG
from utils.MSER import merge_boxes, sort_boxes_reading_order, merge_char_words, filter_char_boxes, get_word_boxes
import os
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [2]:
IMG_SIZE = 128
mser = cv2.MSER_create()

In [90]:
x_non_char_features_hog = []
count = 0

In [ ]:
images_folder = '../data/OCR/coco_non_char/val2017'
for image_name in os.listdir(images_folder):
        if len(x_non_char_features_hog) >= 62000:
                break
        count += 1
        print(len(x_non_char_features_hog))
        # print(f"Processing image: {count}")
        image_path = os.path.join(images_folder, image_name)
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        _, boxes = mser.detectRegions(image)
        unique_boxes = set(tuple(b) for b in boxes)
        unique_boxes = [list(b) for b in unique_boxes]
        unique_boxes = merge_boxes(unique_boxes, threshold=0.3)
        for box in unique_boxes:
                x, y, w, h = box
                char_img = image[y:y+h, x:x+w]
                padded_char_img = cv2.copyMakeBorder(char_img, 5, 5, 5, 5, cv2.BORDER_CONSTANT, value=255)
                blurred_char_img = cv2.GaussianBlur(padded_char_img, (5, 5), 0)
                binarized_char_img = cv2.adaptiveThreshold(
                        blurred_char_img,
                        255,
                        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                        cv2.THRESH_BINARY_INV,
                        11,
                        2
                )
                
                resized_char_img = cv2.resize(
                        binarized_char_img,
                        (128, 128),
                        interpolation=cv2.INTER_CUBIC
                )
                eroded = cv2.erode(resized_char_img, np.ones((5, 5), np.uint8), iterations=2)
                dilated = cv2.dilate(eroded, np.ones((3, 3), np.uint8), iterations=5)
                normalized_char_img = dilated.astype(np.float32) / 255.0
                magnitudes, orientations = calc_gradients(normalized_char_img)
                features = HoG(orientations, magnitudes)
                x_non_char_features_hog.append(features)

301
379
555
726
823
862
890
925
1128
1207
1456
1550
1649
1673
1725
1788
1835
1945
2088
2215
2314
2646
2726
2913
2932
2980
3099
3161
3404
3484
3540
3658
3689
3844
3983
4010
4110
4289
4374
4451
4487
4544
4604
4692
4730
4756
4776
4871
5102
5282
5360
5439
5492
5507
5787
5829
5860
5914
6145
6270
6303
6377
6449
6541
6699
6735
6839
6885
6954
7215
7349
7372
7430
7492
7683
7744
7851
7878
7964
8076
8121
8132
8241
8343
8383
8574
8686
8732
8866
9074
9084
9165
9587
9623
9689
9829
9863
9984
10271
10409
10568
10714
10786
10908
11184
11279
11343
11370
11389
11437
12432
12604
12738
12866
12953
13092
13381
13447
13743
13979
14103
14274
14377
14428
14597
14666
14671
14859
14941
15031
15151
15274
15374
15588
15717
15750
15940
15970
16116
16206
16414
16553
16653
16658
16711
16738
16867
17438
17582
17666
17871
17946
17962
18159
18184
18513
18712
18733
18807
18877
19075
19199
19285
19399
19420
19483
19537
19725
19824
20449
20631
20671
20752
20795
20967
21096
21166
21454
21678
21779
21832
21890
22167
22281
22

In [93]:
joblib.dump(x_non_char_features_hog, './features/x_non_char_features_hog.joblib')

['./features/x_non_char_features_hog.joblib']

In [2]:
x_non_char = joblib.load('./features/x_non_char_features_hog.joblib')
x_char = joblib.load('./features/x_char_features_hog.joblib')

In [3]:
y_non_char = np.zeros(len(x_non_char))
y_char = np.ones(len(x_char))

In [4]:
x = np.concatenate((x_non_char, x_char), axis=0)
y = np.concatenate((y_non_char, y_char), axis=0)


In [5]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [14]:
model = LinearSVC(class_weight='balanced', random_state=42, C=2)


In [15]:
model.fit(x_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",2
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo 

In [16]:
predictions = model.predict(x_test)

In [17]:
f1 = f1_score(y_test, predictions)
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)

In [18]:
print(f"F1 Score: {f1}")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")

F1 Score: 0.9979259731971921
Accuracy: 0.9979290294316779
Precision: 0.9977667889615569
Recall: 0.9980852082336046


In [19]:
joblib.dump(model, './models/is_char_svm_model.joblib')

['./models/is_char_svm_model.joblib']